# Phase 2 Ingestion Walkthrough

This notebook walks through the Phase 2 ingestion path for `ExtendedRunEvent` telemetry snapshots.

It uses the implementation directly:

- `obs_platform.telemetry.v1.load_fixture` loads validated telemetry fixtures.
- `obs_platform.ingestion.runs.ingest_run_event` performs the same persistence work used by the API endpoint.
- SQLAlchemy ORM models in `obs_platform.db.models` inspect the rows written to PostgreSQL.
- `obs_platform.main.create_app` and the `/v1/runs` route demonstrate the HTTP contract without starting a separate server.

Prerequisites from the repository root:

```bash
cp .env.example .env
docker compose up -d --wait postgres
uv run alembic upgrade head
```

## 1. Imports And Notebook Helpers

These helpers are only for the notebook: they create a DB session, clean up the demo run IDs, and display query results compactly. The ingestion behavior itself is imported from the application.

In [ ]:
from collections.abc import AsyncIterator
from dataclasses import asdict, is_dataclass
from datetime import date, datetime
from decimal import Decimal
import json
from pathlib import Path
import sys
from typing import Any

from IPython.display import Markdown, display
from httpx import ASGITransport, AsyncClient
from sqlalchemy import delete, func, select
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from obs_platform.config import DatabaseOnlySettings
from obs_platform.database import create_engine, wait_for_database
from obs_platform.db.models import AgentRun, LLMCall, Span, ToolCall
from obs_platform.ingestion.runs import ingest_run_event
from obs_platform.main import create_app
from obs_platform.routes import runs
from obs_platform.telemetry.v1 import load_fixture


DEMO_RUN_ID = "notebook-phase-2-healthy-success"
DEMO_HITL_RUN_ID = "notebook-phase-2-hitl-lifecycle"


def jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, datetime | date):
        return value.isoformat()
    if isinstance(value, Decimal):
        return float(value)
    return value


def table(rows: list[dict[str, Any]]) -> None:
    if not rows:
        display(Markdown("_No rows._"))
        return

    columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        values = []
        for column in columns:
            value = jsonable(row[column])
            if isinstance(value, dict | list):
                value = json.dumps(value, sort_keys=True)
            values.append(str(value).replace("\n", " ").replace("|", "\\|"))
        body.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join([header, divider, *body])))


def model_dict(instance: Any) -> dict[str, Any]:
    return {
        column.name: jsonable(getattr(instance, column.key))
        for column in instance.__table__.columns
    }


def clone_fixture(name: str, run_id: str):
    event = load_fixture(name)
    event.run_id = run_id
    return event


async def delete_run(session: AsyncSession, run_id: str) -> None:
    for model in (LLMCall, ToolCall, Span, AgentRun):
        await session.execute(delete(model).where(model.run_id == run_id))
    await session.commit()


async def counts_for_run(session: AsyncSession, run_id: str) -> dict[str, int]:
    return {
        "agent_runs": await session.scalar(select(func.count()).where(AgentRun.run_id == run_id)),
        "spans": await session.scalar(select(func.count()).where(Span.run_id == run_id)),
        "tool_calls": await session.scalar(select(func.count()).where(ToolCall.run_id == run_id)),
        "llm_calls": await session.scalar(select(func.count()).where(LLMCall.run_id == run_id)),
    }


settings = DatabaseOnlySettings()
engine = create_engine(settings.db)
await wait_for_database(engine)
Session = async_sessionmaker(engine, expire_on_commit=False)

display(Markdown(f"Connected to PostgreSQL at `{settings.db.host}:{settings.db.port}/{settings.db.name}`."))

## 2. Load A Valid `ExtendedRunEvent`

The fixture loader returns a fully validated Pydantic `ExtendedRunEvent`. The notebook changes only the `run_id` so the demo can be rerun without colliding with canonical fixture names.

In [ ]:
event = clone_fixture("healthy_success", DEMO_RUN_ID)

table([
    {
        "run_id": event.run_id,
        "schema_version": event.schema_version,
        "event_type": event.event_type.value,
        "status": event.status.value,
        "agent_name": event.agent_name,
        "agent_version": event.agent_version,
        "scenario_id": event.scenario_id,
        "spans": len(event.spans),
        "tool_calls": len(event.tool_calls),
        "llm_calls": len(event.llm_calls),
    }
])

Here is the request body shape the API receives. This is the actual fixture serialized through Pydantic's JSON mode.

In [ ]:
payload = event.model_dump(mode="json")
print(json.dumps(payload, indent=2)[:4000])
print("\n... payload truncated for display ...")

## 3. Call `POST /v1/runs`

The route is thin: FastAPI validates the request body as `ExtendedRunEvent`, then calls `ingest_run_event(session, event)`. This cell exercises the real route with an in-process ASGI client and the real database session.

In [ ]:
async with Session() as session:
    await delete_run(session, DEMO_RUN_ID)

    async def notebook_session() -> AsyncIterator[AsyncSession]:
        yield session

    app = create_app()
    app.dependency_overrides[runs.get_session] = notebook_session

    async with AsyncClient(
        transport=ASGITransport(app=app),
        base_url="http://testserver",
    ) as client:
        response = await client.post("/v1/runs", json=payload)

    table([
        {
            "http_status": response.status_code,
            "response_body": response.json(),
        }
    ])
    await session.commit()

## 4. Inspect The Stored Run Row

`agent_runs` stores the run-level fields, flattened HITL fields, final result fields, runtime error fields, and denormalized usage totals derived from persisted child rows.

In [ ]:
async with Session() as session:
    result = await session.execute(
        select(
            AgentRun.run_id,
            AgentRun.event_type,
            AgentRun.status,
            AgentRun.agent_name,
            AgentRun.agent_version,
            AgentRun.prompt_version,
            AgentRun.scenario_id,
            AgentRun.hitl_state,
            AgentRun.usage_total_llm_calls,
            AgentRun.usage_total_tool_calls,
            AgentRun.usage_total_tokens,
            AgentRun.usage_total_retries,
            AgentRun.usage_total_estimated_cost_usd,
            AgentRun.ingested_at,
            AgentRun.updated_at,
        ).where(AgentRun.run_id == DEMO_RUN_ID)
    )
    table([dict(row._mapping) for row in result])

## 5. Inspect Normalized Child Rows

Spans keep the producer's external `span_id`, plus an internal integer `id` used only for database foreign keys. Tool and LLM calls keep their stable external call IDs and point to the internal span row.

In [ ]:
async with Session() as session:
    span_rows = await session.execute(
        select(
            Span.id,
            Span.span_id,
            Span.parent_span_id,
            Span.name,
            Span.sequence,
            Span.status,
        )
        .where(Span.run_id == DEMO_RUN_ID)
        .order_by(Span.sequence)
    )
    tool_rows = await session.execute(
        select(
            ToolCall.tool_call_id,
            ToolCall.span_id,
            ToolCall.tool_name,
            ToolCall.sequence,
            ToolCall.status,
            ToolCall.latency_ms,
            ToolCall.retry_count,
        )
        .where(ToolCall.run_id == DEMO_RUN_ID)
        .order_by(ToolCall.sequence)
    )
    llm_rows = await session.execute(
        select(
            LLMCall.llm_call_id,
            LLMCall.span_id,
            LLMCall.call_type,
            LLMCall.model,
            LLMCall.provider,
            LLMCall.status,
            LLMCall.total_tokens,
            LLMCall.estimated_cost_usd,
        )
        .where(LLMCall.run_id == DEMO_RUN_ID)
        .order_by(LLMCall.started_at)
    )

display(Markdown("### spans"))
table([dict(row._mapping) for row in span_rows])
display(Markdown("### tool_calls"))
table([dict(row._mapping) for row in tool_rows])
display(Markdown("### llm_calls"))
table([dict(row._mapping) for row in llm_rows])

## 6. Example Rows

These are complete ORM rows converted to dictionaries for inspection. Notice how variable payloads land in JSONB fields while IDs, status, timestamps, latency, token counts, and error fields are scalar columns.

In [ ]:
async with Session() as session:
    run = await session.scalar(select(AgentRun).where(AgentRun.run_id == DEMO_RUN_ID))
    tool_call = await session.scalar(
        select(ToolCall)
        .where(ToolCall.run_id == DEMO_RUN_ID)
        .order_by(ToolCall.sequence)
        .limit(1)
    )
    llm_call = await session.scalar(
        select(LLMCall)
        .where(LLMCall.run_id == DEMO_RUN_ID)
        .order_by(LLMCall.started_at)
        .limit(1)
    )

print("agent_runs example row")
print(json.dumps(model_dict(run), indent=2, sort_keys=True))
print("\ntool_calls example row")
print(json.dumps(model_dict(tool_call), indent=2, sort_keys=True))
print("\nllm_calls example row")
print(json.dumps(model_dict(llm_call), indent=2, sort_keys=True))

## 7. Idempotent Re-Ingestion

Phase 2 treats run snapshots as authoritative restatements. Reposting the same `run_id` updates existing rows through `ON CONFLICT DO UPDATE`; it does not create duplicate runs, spans, tool calls, or LLM calls.

In [ ]:
async with Session() as session:
    before = await counts_for_run(session, DEMO_RUN_ID)
    before_span_ids = {
        span_id: internal_id
        for span_id, internal_id in await session.execute(
            select(Span.span_id, Span.id).where(Span.run_id == DEMO_RUN_ID)
        )
    }
    await session.commit()

    result = await ingest_run_event(session, event)

    after = await counts_for_run(session, DEMO_RUN_ID)
    after_span_ids = {
        span_id: internal_id
        for span_id, internal_id in await session.execute(
            select(Span.span_id, Span.id).where(Span.run_id == DEMO_RUN_ID)
        )
    }
    await session.commit()

table([
    {"check": "ingestion_result", "value": result},
    {"check": "row_counts_before", "value": before},
    {"check": "row_counts_after", "value": after},
    {"check": "counts_unchanged", "value": before == after},
    {"check": "span_internal_ids_stable", "value": before_span_ids == after_span_ids},
])

## 8. HITL Pending To Approved Lifecycle

The HITL fixtures show why `/v1/runs` returns `200` and behaves as an upsert endpoint. The pending snapshot creates the run in an awaiting-approval state. The approved final snapshot overwrites the same `agent_runs` row and adds any new post-resume child rows.

In [ ]:
pending = clone_fixture("hitl_pending", DEMO_HITL_RUN_ID)
approved = clone_fixture("hitl_approved", DEMO_HITL_RUN_ID)

async with Session() as session:
    await delete_run(session, DEMO_HITL_RUN_ID)

    pending_result = await ingest_run_event(session, pending)
    pending_counts = await counts_for_run(session, DEMO_HITL_RUN_ID)
    pending_state = await session.execute(
        select(
            AgentRun.event_type,
            AgentRun.status,
            AgentRun.hitl_state,
            AgentRun.hitl_pending_action,
            AgentRun.final_result_output,
        ).where(AgentRun.run_id == DEMO_HITL_RUN_ID)
    )
    pending_span_ids = {
        span_id: internal_id
        for span_id, internal_id in await session.execute(
            select(Span.span_id, Span.id).where(Span.run_id == DEMO_HITL_RUN_ID)
        )
    }
    await session.commit()

    approved_result = await ingest_run_event(session, approved)
    approved_counts = await counts_for_run(session, DEMO_HITL_RUN_ID)
    approved_state = await session.execute(
        select(
            AgentRun.event_type,
            AgentRun.status,
            AgentRun.hitl_state,
            AgentRun.hitl_pending_action,
            AgentRun.final_result_output,
        ).where(AgentRun.run_id == DEMO_HITL_RUN_ID)
    )
    approved_span_ids = {
        span_id: internal_id
        for span_id, internal_id in await session.execute(
            select(Span.span_id, Span.id).where(Span.run_id == DEMO_HITL_RUN_ID)
        )
    }
    await session.commit()

display(Markdown("### lifecycle echo"))
table([
    {"snapshot": "pending", "result": pending_result, "counts": pending_counts},
    {"snapshot": "approved", "result": approved_result, "counts": approved_counts},
])

pending_state_row = dict(pending_state.one()._mapping)
approved_state_row = dict(approved_state.one()._mapping)

display(Markdown("### agent_runs state"))
table([
    {"snapshot": "pending", **pending_state_row},
    {"snapshot": "approved", **approved_state_row},
])

carried_over_span_ids = {span.span_id for span in pending.spans}
stable_carried_over_spans = all(
    approved_span_ids[span_id] == pending_span_ids[span_id]
    for span_id in carried_over_span_ids
)

display(Markdown("### identity checks"))
table([
    {"check": "single evolving agent_runs row", "value": approved_counts["agent_runs"] == 1},
    {"check": "pending_action cleared after approval", "value": approved_state_row["hitl_pending_action"] is None},
    {"check": "final result written after approval", "value": approved_state_row["final_result_output"] is not None},
    {"check": "carried-over span IDs kept internal identity", "value": stable_carried_over_spans},
])

### HITL span tree after approval

This confirms the two carried-over spans kept their internal row identity and the newly added post-approval span points back to the existing root span.

In [ ]:
async with Session() as session:
    rows = (
        await session.execute(
            select(
                Span.id,
                Span.span_id,
                Span.parent_span_id,
                Span.name,
                Span.sequence,
                Span.status,
            )
            .where(Span.run_id == DEMO_HITL_RUN_ID)
            .order_by(Span.sequence, Span.id)
        )
    ).all()

table([dict(row._mapping) for row in rows])

## 9. Cleanup

This removes only the notebook demo run IDs.

In [ ]:
async with Session() as session:
    await delete_run(session, DEMO_RUN_ID)
    await delete_run(session, DEMO_HITL_RUN_ID)

await engine.dispose()
display(Markdown("Notebook demo rows removed and engine disposed."))